# MMVC Trainer - 2. Configuration File Creation

このノートブックでは学習設定ファイルを作成し、データセットを準備します。

## 1. データセットの確認

In [ ]:
import os
import glob
from pathlib import Path

# データセットフォルダの確認
dataset_path = Path('dataset/textful')

for speaker_dir in dataset_path.iterdir():
    if speaker_dir.is_dir():
        print(f"\nSpeaker: {speaker_dir.name}")
        
        # WAVファイルの確認
        wav_files = list(speaker_dir.rglob('*.wav'))
        print(f"  WAV files: {len(wav_files)}")
        
        # transcript.txtの確認
        transcript_file = speaker_dir / 'transcript.txt'
        if transcript_file.exists():
            with open(transcript_file, 'r', encoding='utf-8') as f:
                lines = f.readlines()
            print(f"  Transcript lines: {len(lines)}")
            print(f"  Sample: {lines[0].strip() if lines else 'Empty'}")
        else:
            print(f"  ⚠️ transcript.txt not found!")

## 2. transcript.txtファイルの作成例

各話者フォルダにtranscript.txtファイルが必要です。以下の形式で作成してください：

In [ ]:
# transcript.txtファイルの作成例
sample_transcript = """audio001|こんにちは、今日はいい天気ですね。
audio002|音声変換の学習を始めます。
audio003|日本語の音素変換を行います。
audio004|VITS（ヴィッツ）アーキテクチャを使用しています。
audio005|リアルタイム音声変換システムです。"""

print("transcript.txt example format:")
print(sample_transcript)
print("\nFormat: filename|japanese_text")
print("- filename: WAVファイル名（拡張子なし）")
print("- japanese_text: 対応する日本語テキスト")

In [ ]:
# transcript.txtファイルの自動生成（オプション）
def create_sample_transcript(speaker_dir):
    """WAVファイルに基づいてサンプルtranscript.txtを生成"""
    wav_files = list(Path(speaker_dir).rglob('*.wav'))
    
    if not wav_files:
        print(f"No WAV files found in {speaker_dir}")
        return
    
    transcript_path = Path(speaker_dir) / 'transcript.txt'
    
    sample_texts = [
        "こんにちは、今日はいい天気ですね。",
        "音声変換の学習を始めます。",
        "日本語の音素変換を行います。",
        "VITS（ヴィッツ）アーキテクチャを使用しています。",
        "リアルタイム音声変換システムです。",
        "機械学習を使った音声合成技術です。",
        "高品質な音声データを生成できます。",
        "多話者音声変換に対応しています。"
    ]
    
    with open(transcript_path, 'w', encoding='utf-8') as f:
        for i, wav_file in enumerate(wav_files):
            filename = wav_file.stem
            text = sample_texts[i % len(sample_texts)]
            f.write(f"{filename}|{text}\n")
    
    print(f"Created sample transcript.txt for {speaker_dir} with {len(wav_files)} entries")

# 各話者フォルダに対してサンプルtranscript.txtを生成
# ⚠️ 注意: 実際のテキストに置き換えてください！
generate_sample = input("Generate sample transcript.txt files? (y/n): ")
if generate_sample.lower() == 'y':
    for speaker_dir in dataset_path.iterdir():
        if speaker_dir.is_dir():
            create_sample_transcript(speaker_dir)

## 3. データセットの作成

In [ ]:
# pyopenjtalkのテスト
try:
    import pyopenjtalk
    test_text = "こんにちは、世界"
    phonemes = pyopenjtalk.g2p(test_text)
    print(f"pyopenjtalk test successful: {test_text} -> {phonemes}")
except ImportError:
    print("Installing pyopenjtalk...")
    !pip install pyopenjtalk
    import pyopenjtalk
    test_text = "こんにちは、世界"
    phonemes = pyopenjtalk.g2p(test_text)
    print(f"pyopenjtalk installed and tested: {test_text} -> {phonemes}")

In [ ]:
# データセットの作成
!python create_dataset_jtalk.py --dataset_dir dataset --output_dir filelists --config_template configs/dataset_config.json

## 4. 設定ファイルの確認と調整

In [ ]:
import json

# 生成された設定ファイルの確認
with open('configs/dataset_config.json', 'r', encoding='utf-8') as f:
    config = json.load(f)

print("Generated configuration:")
print(json.dumps(config, indent=2, ensure_ascii=False))

In [ ]:
# 多話者対応の設定調整
def create_multispeaker_config():
    # baseconfig.jsonをコピーして多話者設定を作成
    with open('configs/baseconfig.json', 'r', encoding='utf-8') as f:
        config = json.load(f)
    
    # 話者数の確認
    with open('filelists/train.txt', 'r', encoding='utf-8') as f:
        lines = f.readlines()
    
    speakers = set()
    for line in lines:
        parts = line.strip().split('|')
        if len(parts) > 2:  # マルチスピーカー形式
            speakers.add(int(parts[1]))
    
    n_speakers = len(speakers) if speakers else 0
    
    if n_speakers > 1:
        # マルチスピーカー設定
        config['data']['n_speakers'] = n_speakers
        config['model']['gin_channels'] = 256
        
        # 学習設定を調整
        config['train']['batch_size'] = max(8, 16 // n_speakers)  # GPU使用量を調整
        
        with open('configs/multispeaker_config.json', 'w', encoding='utf-8') as f:
            json.dump(config, f, indent=2, ensure_ascii=False)
        
        print(f"Multi-speaker configuration created:")
        print(f"  - Speakers: {n_speakers}")
        print(f"  - Speaker IDs: {sorted(speakers)}")
        print(f"  - Batch size: {config['train']['batch_size']}")
        print(f"  - Config saved to: configs/multispeaker_config.json")
    else:
        print("Single speaker detected, using base configuration.")
    
    return n_speakers

n_speakers = create_multispeaker_config()

## 5. データセット統計

In [ ]:
# データセット統計の表示
def show_dataset_stats():
    train_file = 'filelists/train.txt'
    val_file = 'filelists/val.txt'
    
    if os.path.exists(train_file):
        with open(train_file, 'r', encoding='utf-8') as f:
            train_lines = f.readlines()
        print(f"Training samples: {len(train_lines)}")
    
    if os.path.exists(val_file):
        with open(val_file, 'r', encoding='utf-8') as f:
            val_lines = f.readlines()
        print(f"Validation samples: {len(val_lines)}")
    
    # 音素長の統計
    if train_lines:
        phoneme_lengths = []
        for line in train_lines[:100]:  # サンプル100行
            parts = line.strip().split('|')
            phonemes = parts[-1]  # 最後の部分が音素
            phoneme_lengths.append(len(phonemes))
        
        print(f"Average phoneme length: {sum(phoneme_lengths)/len(phoneme_lengths):.1f}")
        print(f"Min phoneme length: {min(phoneme_lengths)}")
        print(f"Max phoneme length: {max(phoneme_lengths)}")

show_dataset_stats()

## 設定完了！

次のステップ:
- シングルスピーカー: "3. Train_MMVC.ipynb"を実行
- マルチスピーカー: "3. Train_MMVC.ipynb"でマルチスピーカー設定を使用